### Phase 1.1: Sentinel-5P & MODIS Extraction Script

In [1]:
import ee
import pandas as pd
import time

# ---------------------------------------------------------
# 1. INITIALIZATION & AUTHENTICATION
# ---------------------------------------------------------
# Trigger the authentication flow. (Run this once per session)
try:
    ee.Initialize()
except Exception as e:
    ee.Authenticate()
    ee.Initialize()

# ---------------------------------------------------------
# 2. MASTER PARAMETERS
# ---------------------------------------------------------
# Delhi-NCR Bounding Box (Approximate full NCR extent)
# Format: [min_lon, min_lat, max_lon, max_lat]
BBOX = [76.15, 27.50, 78.40, 29.20]
ROI = ee.Geometry.Rectangle(BBOX)

START_YEAR = 2018
END_YEAR = 2025

# Spatial resolution for export (in meters). 
# 1000m roughly equals 0.01 degrees. We will enforce the exact 141x231 
# matrix in Phase 1.3, so getting a clean 1km resolution here is perfect.
SCALE = 1000 
CRS = 'EPSG:4326'

# ---------------------------------------------------------
# 3. GEE DATASET MAPPINGS
# ---------------------------------------------------------
DATASETS = {
    'NO2': {
        'collection': 'COPERNICUS/S5P/OFFL/L3_NO2',
        'band': 'tropospheric_NO2_column_number_density'
    },
    'CO': {
        'collection': 'COPERNICUS/S5P/OFFL/L3_CO',
        'band': 'CO_column_number_density'
    },
    'AER_AI': {
        'collection': 'COPERNICUS/S5P/OFFL/L3_AER_AI',
        'band': 'absorbing_aerosol_index'
    },
    'AOD': {
        'collection': 'MODIS/061/MCD19A2_GRANULES',
        'band': 'Optical_Depth_047'
    }
}

# ---------------------------------------------------------
# 4. HELPER FUNCTIONS
# ---------------------------------------------------------
def get_daily_image(date, collection_name, band_name, roi):
    """Fetches the mean image for a specific date."""
    start_date = ee.Date(date)
    end_date = start_date.advance(1, 'day')
    
    collection = (ee.ImageCollection(collection_name)
                  .filterBounds(roi)
                  .filterDate(start_date, end_date)
                  .select(band_name))
    
    # Calculate daily mean. If no data (e.g., cloudy/missing pass), 
    # GEE returns a masked image.
    daily_mean = collection.mean().rename(ee.String(date).replace('-', '_', 'g'))
    return daily_mean.set('system:time_start', start_date.millis())

# ---------------------------------------------------------
# 5. EXECUTION & YEARLY BATCHING
# ---------------------------------------------------------
print("Starting Phase 1.1 GEE Extraction Pipeline...")

for year in range(START_YEAR, END_YEAR + 1):
    # Generate list of dates for the current year
    start_date = f"{year}-01-01"
    end_date = f"{year}-12-31"
    dates = pd.date_range(start=start_date, end=end_date, freq='D').strftime('%Y-%m-%d').tolist()
    
    for var_name, var_info in DATASETS.items():
        print(f"Processing {var_name} for the year {year}...")
        
        # We construct an ee.List of dates and map our daily function over it
        ee_dates = ee.List(dates)
        
        # Create an ImageCollection of daily means, then convert to a single 
        # multi-band image (365/366 bands) for easy export
        daily_collection = ee.ImageCollection.fromImages(
            ee_dates.map(lambda d: get_daily_image(d, var_info['collection'], var_info['band'], ROI))
        )
        
        # Stack all daily bands into a single image
        yearly_stack = daily_collection.toBands()
        
        # Define export task to Google Drive
        task_name = f"Delhi_NCR_{var_name}_{year}"
        task = ee.batch.Export.image.toDrive(
            image=yearly_stack.clip(ROI),
            description=task_name,
            folder='Delhi_NCR_Air_Quality_Project',
            fileNamePrefix=task_name,
            region=ROI.getInfo()['coordinates'],
            scale=SCALE,
            crs=CRS,
            maxPixels=1e10
        )
        
        # Start the task
        task.start()
        print(f"  -> Task '{task_name}' submitted to Earth Engine servers.")
        
        # Sleep briefly to avoid hammering the API request limit
        time.sleep(2)

print("\nAll tasks submitted! Check https://code.earthengine.google.com/tasks to monitor progress.")

/home/yashaswi-garg/anaconda3/envs/aerosense/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


Enter verification code:  4/1Aci98E_0vhb4vsCIPjuUF9lyfdQApPzfQTQ9PcM-9sDRbToMFoQNUy_mVkY



Successfully saved authorization token.
Starting Phase 1.1 GEE Extraction Pipeline...
Processing NO2 for the year 2018...
  -> Task 'Delhi_NCR_NO2_2018' submitted to Earth Engine servers.
Processing CO for the year 2018...
  -> Task 'Delhi_NCR_CO_2018' submitted to Earth Engine servers.
Processing AER_AI for the year 2018...
  -> Task 'Delhi_NCR_AER_AI_2018' submitted to Earth Engine servers.
Processing AOD for the year 2018...
  -> Task 'Delhi_NCR_AOD_2018' submitted to Earth Engine servers.
Processing NO2 for the year 2019...
  -> Task 'Delhi_NCR_NO2_2019' submitted to Earth Engine servers.
Processing CO for the year 2019...
  -> Task 'Delhi_NCR_CO_2019' submitted to Earth Engine servers.
Processing AER_AI for the year 2019...
  -> Task 'Delhi_NCR_AER_AI_2019' submitted to Earth Engine servers.
Processing AOD for the year 2019...
  -> Task 'Delhi_NCR_AOD_2019' submitted to Earth Engine servers.
Processing NO2 for the year 2020...
  -> Task 'Delhi_NCR_NO2_2020' submitted to Earth Eng